# 12 因果推論與政策評估

淋浴暴露真的「導致」感染嗎？消毒水系統真的有效嗎？

流程：**DAG 因果圖 → 干擾/中介/碰撞 → 歸因風險 AR/PAR → DiD 介入評估 → 平行趨勢檢驗**

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

## Step 1 — DAG：用文字/圖形釐清因果結構，不需要 graphviz

因果推論的第一步不是跑迴歸，而是**先畫出你認為的因果關係**。這裡用 matplotlib 手畫節點和箭頭，不需要另外安裝 graphviz（在 Colab 或 CI 常常裝不起來，多一個系統相依套件就多一個出錯點）。畫完圖之後，關鍵是辨認三種角色：**干擾因子**（confounder，同時影響暴露與結果）、**中介變項**（mediator，暴露透過它才影響結果）、**碰撞因子**（collider，被兩個變項同時指向）。

> **逐行拆解**：
>
> | 這行程式 | 在做什麼 |
> |---|---|
> | `nodes = {...}` | 用座標字典定義 DAG 中每個變項的方框位置 |
> | `for src, dst in arrows: ax.annotate(...)` | 依因果方向畫箭頭；箭頭指向誰，誰就是「果」 |
> | `print("干擾因子：functional_status → shower_use 且 → infection")` | 辨識干擾因子：同時指向暴露與結果的變項 |
> | `print("碰撞因子：hospitalized ← severity 且 ← infection")` | 辨識碰撞因子：被兩個箭頭同時指入的變項，校正它反而製造假關聯 |

> 🧭 **三種角色，兩種處理方式完全相反**：干擾因子要校正（Ch05 分層分析、Ch06 迴歸校正都是在做這件事）；中介變項通常不校正（校正了就把因果路徑本身「關掉」，會低估總效果）；碰撞因子絕對不能校正——校正碰撞因子會打開一條原本不存在的路徑，產生假性關聯（collider bias）。


In [ ]:
# --- Step 1: DAG（有向無環圖）---
# 用文字描述因果關係（不需要安裝 graphviz）
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.patches as mpatches
import statsmodels.formula.api as smf

# -- CJK font setup (避免中文標籤顯示為方框) --
# 掃描系統字型目錄，顯式註冊 CJK 字型（比依賴快取更可靠）
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

# DAG 視覺化
fig, ax = plt.subplots(figsize=(10, 6))
ax.set_xlim(0, 10)
ax.set_ylim(0, 7)
ax.axis("off")

# 節點
nodes = {
    "floor/wing": (1, 5.5),
    "water\ncontamination": (3.5, 5.5),
    "shower\naerosol": (6, 5.5),
    "infection": (8.5, 5.5),
    "functional\nstatus": (1, 3),
    "shower\nuse": (4, 3),
    "age": (1, 1),
    "comorbidities": (3.5, 1),
    "severity": (6, 1),
    "death": (8.5, 1),
}

for name, (x, y) in nodes.items():
    ax.add_patch(plt.Rectangle((x-0.7, y-0.4), 1.4, 0.8,
                 fill=True, facecolor="#e0e0e0", edgecolor="black", linewidth=1.5))
    ax.text(x, y, name, ha="center", va="center", fontsize=8, fontweight="bold")

# 箭頭（因果方向）
arrows = [
    ("floor/wing", "water\ncontamination"),
    ("water\ncontamination", "shower\naerosol"),
    ("shower\naerosol", "infection"),
    ("functional\nstatus", "shower\nuse"),
    ("shower\nuse", "infection"),
    ("functional\nstatus", "infection"),
    ("age", "comorbidities"),
    ("comorbidities", "severity"),
    ("severity", "death"),
    ("infection", "severity"),
]

for src, dst in arrows:
    x1, y1 = nodes[src]
    x2, y2 = nodes[dst]
    ax.annotate("", xy=(x2-0.7, y2), xytext=(x1+0.7, y1),
                arrowprops=dict(arrowstyle="->", color="#333", lw=1.5))

ax.set_title("Legionella DAG \u2014 因果關係圖", fontsize=14)
plt.tight_layout()
plt.show()

print("=== 因果結構辨識 ===")
print("干擾因子：functional_status \u2192 shower_use 且 \u2192 infection")
print("中介變項：shower_aerosol 在 water_contamination \u2192 infection 之間")
print("碰撞因子：hospitalized \u2190 severity 且 \u2190 infection")
print("\n\u2192 控制干擾因子（Ch05 已做）= 正確")
print("\u2192 控制碰撞因子 = 錯誤！會產生假性關聯")

## Step 2 — 歸因風險（AR）：暴露多背了多少風險？

Attributable Risk（AR）回答的是「暴露組」跟「非暴露組」的絕對風險差了多少——注意是**相減**，不是相除（相除是風險比 RR，Ch03 已經算過）。這裡沿用 Ch05/Ch06 的 `shower_use` 暴露分組，重新算一次侵襲率，再相減。

> **逐行拆解**：
>
> | 這行程式 | 在做什麼 |
> |---|---|
> | `exposed = df[df["shower_use"] == 1]` | 篩出「有淋浴」的暴露組 |
> | `risk_exposed = exposed["infected"].mean()` | 暴露組的侵襲率（0/1 欄位取平均 = 比例） |
> | `AR = risk_exposed - risk_unexposed` | 歸因風險：暴露組比非暴露組多出來的絕對風險 |
> | `PAR = risk_total - risk_unexposed` | 母體歸因風險：把分母換成全體，估這個暴露對整個族群貢獻了多少風險 |

> 💡 **AR 和 RR 是同一份資料的兩種問法**：RR 回答「風險變成幾倍」，AR 回答「多了多少個百分點」。兩者在公衛決策上意義不同——RR 大但 AR 小，代表雖然相對風險很高，實際能預防的病例數卻有限（罕見病常見的情況）。


In [ ]:
# --- Step 2: 歸因風險（Attributable Risk）---
df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

# 淋浴暴露的侵襲率
exposed = df[df["shower_use"] == 1]
unexposed = df[df["shower_use"] == 0]

risk_exposed = exposed["infected"].mean()
risk_unexposed = unexposed["infected"].mean()
risk_total = df["infected"].mean()

print("=== 淋浴暴露的侵襲率 ===")
print(f"淋浴者侵襲率：{risk_exposed:.1%} ({exposed['infected'].sum()}/{len(exposed)})")
print(f"非淋浴者侵襲率：{risk_unexposed:.1%} ({unexposed['infected'].sum()}/{len(unexposed)})")
print(f"全體侵襲率：{risk_total:.1%}")

# Attributable Risk
AR = risk_exposed - risk_unexposed
print(f"\n=== Attributable Risk (AR) ===")
print(f"AR = {risk_exposed:.3f} - {risk_unexposed:.3f} = {AR:.3f}")
print(f"\u2192 淋浴者比非淋浴者多 {AR:.1%} 的感染風險")

# Population Attributable Risk
PAR = risk_total - risk_unexposed
PAR_pct = PAR / risk_total * 100
print(f"\n=== Population Attributable Risk (PAR) ===")
print(f"PAR = {risk_total:.3f} - {risk_unexposed:.3f} = {PAR:.3f}")
print(f"PAR% = {PAR_pct:.1f}%")
print(f"\u2192 如果消除淋浴暴露，理論上可減少 {PAR_pct:.0f}% 的感染")
print("\u2192 前提：因果關係成立，且無其他傳播途徑")

## Step 2b — PAF 兩種算法：Levin 公式 vs 直接相減，結果必須一致

上一步的 PAR 是用「風險」相減算出來的絕對版本；這裡改用 **Levin 公式**，只需要暴露盛行率 `Pe` 和風險比 `RR` 兩個數字，就能算出**族群歸因分數**（Population Attributable Fraction, PAF）——也就是 PAR 除以全體風險後的百分比版本。兩種算法理論上完全等價，這裡直接算出來互相驗證。

> **逐行拆解**：
>
> | 這行程式 | 在做什麼 |
> |---|---|
> | `Pe = (df["shower_use"] == 1).mean()` | 暴露盛行率：全體住民中有淋浴的比例 |
> | `RR = risk_exposed / risk_unexposed` | 風險比，沿用 Step 2 算好的兩個風險 |
> | `PAF_levin = Pe * (RR - 1) / (1 + Pe * (RR - 1))` | Levin 公式：只用 Pe 和 RR 就能推出族群歸因分數 |
> | `PAF_alt = (risk_total - risk_unexposed) / risk_total` | 另一種等價算法：直接用「全體風險」與「未暴露風險」的相對差 |

> 🧭 PAF = 「如果整個族群都不暴露，理論上可減少的病例比例」——**只有因果成立時才這樣解讀**；如果 `shower_use` 只是跟真正病因（例如水系統汙染）一起出現的旁觀者，消除淋浴並不會讓 PAF% 的病例真的消失。


In [ ]:
# --- Step 2b: 族群歸因分數 PAF（Levin 公式 vs I_total 形式）---
# 族群歸因分數 PAF：Levin 公式 vs (I_total - I_unexposed)/I_total —— 兩式等價
Pe = (df["shower_use"] == 1).mean()          # 暴露盛行率
RR = risk_exposed / risk_unexposed
PAF_levin = Pe * (RR - 1) / (1 + Pe * (RR - 1))
risk_total = df["infected"].mean()
PAF_alt = (risk_total - risk_unexposed) / risk_total

print("=== 族群歸因分數 PAF（兩種算法互相驗證）===")
print(f"暴露盛行率 Pe = {Pe:.1%}，風險比 RR = {RR:.2f}")
print(f"PAF (Levin)      = {PAF_levin:.1%}")
print(f"PAF (I_tot 形式) = {PAF_alt:.1%}   兩式一致 = {abs(PAF_levin - PAF_alt) < 1e-9}")

## Step 2c — 換一個乾淨的例子：食物中毒 2×2 表（教學示範數字）

legionella 資料有 280 筆、訊號偏弱，AR/PAF 的數字不夠「漂亮」。這裡換一組**教學示範數字**（非真實研究、非真實事件），用最經典的 2×2 表格式練習同一套公式，確認你真的會算、不是只會複製貼上。

> **逐行拆解**：
>
> | 這行程式 | 在做什麼 |
> |---|---|
> | `a, b, c, d = 120, 280, 25, 375` | 教學示範數字：a=暴露且患病、b=暴露未患病、c=未暴露且患病、d=未暴露未患病 |
> | `risk_exposed_fp = a / (a + b)` | 暴露組患病風險 |
> | `risk_unexposed_fp = c / (c + d)` | 未暴露組患病風險 |
> | `Pe_fp = (a + b) / (a + b + c + d)` | 暴露盛行率 |
> | `PAF_fp = Pe_fp * (RR_fp - 1) / (1 + Pe_fp * (RR_fp - 1))` | 套用同一條 Levin 公式 |

> ⚠️ 這是**教學示範數字**，不是任何真實食物中毒事件的統計——選這組數字純粹是因為它算出來的 AR≈23.7%、RR≈4.80、PAF≈65.5%，剛好是好記的量級，方便驗算。真實世界的疫情資料請一律回到 `legionella_outbreak.csv`。


In [ ]:
# --- Step 2c: 教學示範 2x2 表 — 食物中毒 AR/RR/PAF（非真實研究，教學示範數字）---
# 2x2 table: a=暴露+患病, b=暴露+未患病, c=未暴露+患病, d=未暴露+未患病
a, b, c, d = 120, 280, 25, 375  # 教學示範數字，非真實事件

risk_exposed_fp = a / (a + b)        # 暴露組患病風險
risk_unexposed_fp = c / (c + d)      # 未暴露組患病風險
AR_fp = risk_exposed_fp - risk_unexposed_fp
RR_fp = risk_exposed_fp / risk_unexposed_fp
Pe_fp = (a + b) / (a + b + c + d)    # 暴露盛行率
PAF_fp = Pe_fp * (RR_fp - 1) / (1 + Pe_fp * (RR_fp - 1))

print("=== 教學示範：食物中毒 2x2 表（非真實研究）===")
print(f"暴露組風險 = {a}/{a + b} = {risk_exposed_fp:.3f}")
print(f"未暴露組風險 = {c}/{c + d} = {risk_unexposed_fp:.3f}")
print(f"AR  = {AR_fp:.3f}  ({AR_fp:.1%})")
print(f"RR  = {RR_fp:.2f}")
print(f"PAF = {PAF_fp:.1%}")

## Step 3 — 反事實思考：「如果沒有暴露」會發生什麼？

PAR 已經隱含了反事實邏輯，這裡把它拆開來寫得更明白：假設**全體住民都跟未暴露組有一樣的風險**，理論上會有幾個人感染？跟實際感染人數的差距，就是「理論上可預防」的病例數。

> **逐行拆解**：
>
> | 這行程式 | 在做什麼 |
> |---|---|
> | `counterfactual_cases = int(n_total * risk_unexposed)` | 反事實情境：假設全體都套用未暴露組的風險，預期感染人數 |
> | `prevented = n_infected - counterfactual_cases` | 實際感染數與反事實預期數的差距 = 理論上可預防的病例數 |

> ⚠️ 反事實估算永遠是「what if」，不是「已發生的事實」——前提依然是暴露與結果之間為因果關係，而且完全消除暴露（例如全面禁止住民淋浴）在現實中往往不可行、也不合乎人道。更實際的做法是像 Step 4 開始的 DiD 分析那樣，改善暴露的「安全性」（例如消毒水系統），而不是禁止暴露本身。


In [ ]:
# --- Step 3: 反事實思考 ---
n_total = len(df)
n_infected = df["infected"].sum()

# 反事實：如果所有人都不淋浴
counterfactual_cases = int(n_total * risk_unexposed)
prevented = n_infected - counterfactual_cases

print("=== 反事實情境 ===")
print(f"實際感染人數：{n_infected}")
print(f"如果所有人都不淋浴（反事實）：預期 {counterfactual_cases} 人感染")
print(f"可預防的感染數：{prevented}")
print(f"\n\u2192 但這只是理論估算！")
print("\u2192 實際上，禁止所有人淋浴不可行")
print("\u2192 更實際的做法：消毒水系統，讓淋浴變安全")

## Step 4 — DiD 資料準備：把病例攤成「兩組 × 每日」的長格式面板

DiD（Difference-in-Differences）需要的資料形狀跟前面幾步不一樣：不是一人一列，而是**「介入組/對照組」×「每一天」**的長格式面板（panel）。這裡先把病例依日期分組彙總，缺的日期補 0（沒有病例不代表沒有資料、不能直接刪掉），再組合出 `treated`、`post` 兩個 DiD 迴歸不可或缺的二元欄位。

> **逐行拆解**：
>
> | 這行程式 | 在做什麼 |
> |---|---|
> | `treated_cases = cases[(cases["floor"].isin([2, 3])) & (cases["wing"] == "B")]` | 定義介入組：2-3F B 翼（水系統消毒的目標區域） |
> | `.groupby("symptom_onset_date").size().reindex(all_dates, fill_value=0)` | 依日期彙總病例數，缺的日期補 0，確保每一天都有一列 |
> | `"treated": [1] * len(all_dates) + [0] * len(all_dates)` | 組別標記：1=介入組、0=對照組 |
> | `panel["post"] = (panel["date"] >= "2026-01-25").astype(int)` | 時間標記：1=介入後、0=介入前，跟 `treated` 是 DiD 迴歸的兩根柱子 |
> | `panel["day"] = (panel["date"] - panel["date"].min()).dt.days` | 把日期轉成數值天數，方便畫圖或做時間趨勢分析 |

> 🧭 DiD 的核心資料結構就是這張面板：每一列是「某一組、某一天」的病例數，`treated` 和 `post` 兩個二元欄位交叉起來剛好四種組合（介入組/對照組 × 介入前/介入後），Step 6 的迴歸就是在比較這四格的平均值。


In [ ]:
# --- Step 4: DiD 資料準備 ---
# 情境：1月25日對 2-3F B翼 實施水系統緊急消毒
# 介入組：2-3F B翼（高侵襲率，靠近汙染源）
# 對照組：1F 全部 + 2-3F A翼（不同水源或非目標區）

df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
cases = df[df["infected"] == 1].copy()

# 建立每日面板資料
all_dates = pd.date_range("2026-01-12", "2026-01-28", freq="D")

# 介入組：2-3F B翼
treated_cases = cases[(cases["floor"].isin([2, 3])) & (cases["wing"] == "B")]
# reindex 補 0：沒有病例的日期不能直接消失，DiD 需要每一天都有一列
treated_daily = treated_cases.groupby("symptom_onset_date").size().reindex(all_dates, fill_value=0)

# 對照組：其餘區域
control_cases = cases[~((cases["floor"].isin([2, 3])) & (cases["wing"] == "B"))]
control_daily = control_cases.groupby("symptom_onset_date").size().reindex(all_dates, fill_value=0)

# 組合成長格式面板：「組別 x 每一天」各一列
panel = pd.DataFrame({
    "date": list(all_dates) * 2,
    "treated": [1] * len(all_dates) + [0] * len(all_dates),  # 1=介入組, 0=對照組
    "daily_cases": list(treated_daily.values) + list(control_daily.values),
})
# post 和 treated 是 DiD 迴歸（Step 6）不可或缺的兩個二元變項
panel["post"] = (panel["date"] >= "2026-01-25").astype(int)  # 1=介入後, 0=介入前
panel["day"] = (panel["date"] - panel["date"].min()).dt.days  # 數值化天數，方便畫圖/時間趨勢

print("=== DiD 面板資料 ===")
print(f"介入組（2-3F B翼）：{len(treated_daily)} 天")
print(f"對照組（其餘區域）：{len(control_daily)} 天")
print(f"介入日：2026-01-25")
print(f"\n介入前後病例數：")
summary = panel.groupby(["treated", "post"]).agg(total=('daily_cases','sum'), mean=('daily_cases','mean')).reset_index()
summary["group"] = summary["treated"].map({1: "介入組(2-3F B)", 0: "對照組"})
summary["period"] = summary["post"].map({0: "介入前", 1: "介入後"})
print(summary[["group", "period", "total", "mean"]].to_string(index=False))

## Step 5 — 平行趨勢檢驗：DiD 能不能信，先看圖

DiD 估計值只有在**平行趨勢假設**（parallel trends assumption）成立時才可信：如果介入前兩組本來就走不同的斜率，介入後的差異可能只是兩組原本就在分道揚鑣，跟介入本身無關。畫圖是檢查這個假設最直覺的方法——先看，再信模型。

> **逐行拆解**：
>
> | 這行程式 | 在做什麼 |
> |---|---|
> | `ax.plot(all_dates, treated_daily.values, ...)` | 畫出介入組每日病例數的時間趨勢線 |
> | `ax.plot(all_dates, control_daily.values, ...)` | 畫出對照組的時間趨勢線，放在同一張圖比較 |
> | `ax.axvline(x=pd.Timestamp("2026-01-25"), ...)` | 標出介入日，把時間軸切成「介入前（左側，用來檢驗平行趨勢）」與「介入後（右側，觀察效果）」 |

> ⚠️ 只看介入線**左側**：兩條線如果大致平行上下起伏（就算數值不同也沒關係），平行趨勢假設就站得住腳；如果介入前兩條線就已經一條在漲一條在跌，DiD 估計值就不能簡單解讀成「介入的效果」。


In [ ]:
# --- Step 5: 平行趨勢檢驗 + DiD 視覺化 ---
fig, ax = plt.subplots(figsize=(10, 5))

# 平行趨勢檢驗（parallel trends check）：
# 只看介入日（虛線）左側 — 介入組與對照組兩條線是否大致平行上下起伏
# 若介入前就不平行，代表兩組本來就有不同的變化速度，DiD 估計會失真

# 介入組
ax.plot(all_dates, treated_daily.values, marker="o", markersize=4,
        label="介入組 (2-3F B翼)", color="#e34a33")
# 對照組
ax.plot(all_dates, control_daily.values, marker="s", markersize=4,
        label="對照組 (其餘)", color="#2c7fb8")

# 介入線：把時間軸切成「介入前（左側，用來檢驗平行趨勢）」與「介入後（右側，觀察效果）」
ax.axvline(x=pd.Timestamp("2026-01-25"), color="black", linestyle="--",
           alpha=0.7, label="介入日 (1/25)")

ax.set_title("DiD — 介入前後病例數趨勢")
ax.set_xlabel("日期")
ax.set_ylabel("每日病例數")
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print("→ 觀察介入前（虛線左側）兩條線是否大致平行")
print("→ 如果平行，DiD 估計較可信")

## Step 6 — DiD OLS：`treated:post` 交互作用項才是答案

DiD 迴歸看起來只是一行 `smf.ols(...)`，但重點全部藏在**交互作用項** `treated:post` 裡。`treated` 和 `post` 各自只是控制「組別」和「時間」本身的固定差異，只有 `treated:post` 抓到的是「介入組在介入後，比『組別差異』和『時間趨勢』能解釋的多出（或少掉）多少」——這才是 DiD 真正估計的介入效果。

> **逐行拆解**：
>
> | 這行程式 | 在做什麼 |
> |---|---|
> | `smf.ols("daily_cases ~ treated + post + treated:post", data=panel)` | `treated`=組別主效果、`post`=時間主效果、`treated:post`=交互作用項（DiD 估計值） |
> | `.fit(cov_type="HC3")` | 用 HC3 穩健標準誤，因應面板資料常見的變異數不齊一（heteroskedasticity），避免顯著性檢定過度樂觀 |
> | `did_effect = model.params["treated:post"]` | 取出交互作用項係數，就是「介入對介入組造成的額外效果」 |
> | `did_p = model.pvalues["treated:post"]` | 這個效果是否統計顯著，看的也是 `treated:post` 這一項，不是 `treated` 或 `post` |

> 💡 看 DiD 迴歸結果時，眼睛只要盯緊 `treated:post` 這一行——`treated` 和 `post` 的係數通常不是我們關心的重點，只是模型拿來「扣掉」組別和時間各自的固定差異，讓交互作用項乾淨地代表介入效果。


In [ ]:
# --- Step 6: DiD OLS 迴歸 ---
# treated      → 組別主效果（介入組 vs 對照組，不管時間）
# post         → 時間主效果（介入前 vs 介入後，不管組別）
# treated:post → 交互作用項 = DiD 估計值：介入組在介入後「額外多出（或少掉）」的變化
# cov_type="HC3"：面板資料常見變異數不齊一（heteroskedasticity），用穩健標準誤避免 p-value 過度樂觀
model = smf.ols("daily_cases ~ treated + post + treated:post", data=panel).fit(cov_type="HC3")

print("=== DiD 迴歸結果（HC3 穩健標準誤）===")
print(model.summary().tables[1])

# 只看 treated:post 這一項 — 這才是 DiD 真正的介入效果估計值
did_effect = model.params["treated:post"]
did_p = model.pvalues["treated:post"]

print(f"\n=== DiD 效果估計 ===")
print(f"treated:post 係數 = {did_effect:.3f}")
print(f"p-value = {did_p:.4f}")

if did_effect < 0:
    print(f"\n→ 介入後，介入組每日病例數比預期減少 {abs(did_effect):.1f} 人")
else:
    print(f"\n→ 介入後，介入組每日病例數比預期增加 {did_effect:.1f} 人")

if did_p < 0.05:
    print("→ 效果統計顯著（p < 0.05）")
else:
    print("→ 效果未達統計顯著（p ≥ 0.05）")
    print("→ 可能原因：樣本量不足、觀察期太短、介入效果需要更長時間顯現")

## 小結

| 步驟 | 學到的技能 |
|------|------------|
| DAG | 用圖形辨識干擾、中介、碰撞因子 |
| AR | 淋浴者相對非淋浴者多出的絕對感染風險 |
| PAR / PAF | Levin 公式 vs I_total 形式互相驗證，量化「消除暴露理論上可減少的病例比例」 |
| 2×2 教學範例 | 用食物中毒教學示範數字（非真實研究）練習 AR/RR/PAF 計算 |
| 反事實 | 估算「如果消除暴露」的預期效果 |
| DiD | `daily_cases ~ treated + post + treated:post`（HC3 穩健標準誤） |
| 平行趨勢 | 介入前兩組趨勢是否一致 |

**結論**：
- DAG 幫我們釐清哪些變項該控制、哪些不該控制
- AR/PAR/PAF 量化暴露的貢獻，但前提是因果關係成立
- DiD 是評估介入效果的準實驗方法，但需要平行趨勢假設
- 在觀察性資料中，因果推論永遠需要謹慎

下一章（Ch13），我們確保所有分析可重現 → 可重現研究。